#Use our fine-tuned version of Parler-TTS

In [ ]:
import locale

def getpreferredencoding(do_setlocale=True):
    return "UTF-8"

locale.getpreferredencoding = getpreferredencoding

In [ ]:
!pip install indic-transliteration transformers torch
!pip install git+https://github.com/huggingface/parler-tts.git

  Cloning https://github.com/huggingface/parler-tts.git to /tmp/pip-req-build-9zerf__w
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/parler-tts.git /tmp/pip-req-build-9zerf__w
  Resolved https://github.com/huggingface/parler-tts.git to commit d108732cd57788ec86bc857d99a6cabd66663d68
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/descriptinc/audiotools to /tmp/pip-install-ldci_gzw/descript-audiotools_007a5a0293054819a969dc3431a4da93
  Running command git clone --filter=blob:none --quiet https://github.com/descriptinc/audiotools /tmp/pip-install-ldci_gzw/descript-audiotools_007a5a0293054819a969dc3431a4da93
  Resolved https://github.com/descriptinc/audiotools to commit 348ebf2034ce24e2a91a553e3171cb00c0c71678
  Preparing metadata (setup.py) ... done


##Load Model

In [ ]:
from indic_transliteration.sanscript import transliterate, DEVANAGARI, ITRANS
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
import torch

# Device setup
device = "cuda:0" if torch.cuda.is_available() else "cpu"

print(device)
model = ParlerTTSForConditionalGeneration.from_pretrained("En1gma02/Parler-TTS-Mini-v0.1-Indian-Male-Accent-Hindi-Kaggle").to(device)
tokenizer = AutoTokenizer.from_pretrained("parler-tts/parler_tts_mini_v0.1")


cuda:0


/usr/local/lib/python3.11/dist-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
  "_name_or_path": "google/flan-t5-base",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2048,
  "d_kv": 64,
  "d_model": 768,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 12,
  "num_heads": 12,
  "num_layers": 12,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_pena

##Generate Audio - Hindi

In [ ]:
# Input text in Devanagari
prompt_in_devanagari = "हम अपने संकाय के समक्ष पार्लर टेक्स्ट टू स्पीच की प्रस्तुति कर रहे हैं।"
description = "'Male Hindi Speaker speaks his words slowly and clearly, in a very calm sounding environment with clear audio quality. He speaks fast and in Hindi.'"

# Transliterate the Devanagari prompt to Latin script (ABCD)
prompt_transliterated = transliterate(prompt_in_devanagari, DEVANAGARI, ITRANS)

print("Prompt: ", prompt_in_devanagari)
print("Prompt Transliterated: ", prompt_transliterated)
# Tokenize inputs
input_ids = tokenizer(description, return_tensors="pt").input_ids.to(device)
prompt_input_ids = tokenizer(prompt_transliterated, return_tensors="pt").input_ids.to(device)

# Generate audio
generation = model.generate(input_ids=input_ids, prompt_input_ids=prompt_input_ids)
audio_arr = generation.cpu().numpy().squeeze()

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Prompt:  हम अपने संकाय के समक्ष पार्लर टेक्स्ट टू स्पीच की प्रस्तुति कर रहे हैं।
Prompt Transliterated:  hama apane saMkAya ke samakSha pArlara TeksTa TU spIcha kI prastuti kara rahe haiM|


In [ ]:
from IPython.display import Audio
Audio(audio_arr, rate=model.config.sampling_rate)

##Generate Audio - Indian Accent

In [ ]:
prompt = "Hello guys, this is me trying to speak in indian accent."
description = "'Akshansh delivers his words quite expressively, in a very clear audio quality. He speaks very slowly.'"

input_ids = tokenizer(description, return_tensors="pt").input_ids.to(device)
prompt_input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

generation = model.generate(input_ids=input_ids, prompt_input_ids=prompt_input_ids)
audio_arr = generation.cpu().numpy().squeeze()

##Listen to Audio

In [ ]:
from IPython.display import Audio
Audio(audio_arr, rate=model.config.sampling_rate)

##Save Audio

In [ ]:
import soundfile as sf
import numpy as np

# Convert audio array to float32
audio_arr_float32 = audio_arr.astype(np.float32)

# Specify the file path where you want to save the audio
output_path = 'generated_audio4.wav'

# Write the converted audio array to a WAV file
sf.write(output_path, audio_arr_float32, model.config.sampling_rate)

print(f"Audio saved to {output_path}")

Audio saved to generated_audio4.wav
